# DAY 4 — Fund Performance Analytics
## Mutual Fund Analytics | India | 2022–2026
---

## ⚙️ Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings, os

warnings.filterwarnings('ignore')
os.makedirs('reports', exist_ok=True)

# Go to project root if running from notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f"📁 Working directory: {os.getcwd()}")

proc = 'data/processed/'

# Load datasets
nav         = pd.read_csv(proc + '02_nav_history.csv')
fund_master = pd.read_csv(proc + '01_fund_master.csv')
perf        = pd.read_csv(proc + '07_scheme_performance.csv')
benchmark   = pd.read_csv(proc + '10_benchmark_indices.csv')

# Parse dates
nav['date']       = pd.to_datetime(nav['date'])
benchmark['date'] = pd.to_datetime(benchmark['date'])

# ✅ Auto-detect and fix swapped nav/amfi_code columns
# amfi_code should be large integers (100000–200000), NAV should be small floats (10–5000)
sample_nav  = pd.to_numeric(nav['nav'].head(20),      errors='coerce').mean()
sample_amfi = pd.to_numeric(nav['amfi_code'].head(20), errors='coerce').mean()
if sample_nav > 10000:  # nav column has amfi codes (large numbers) → columns are swapped
    print("⚠️  Detected swapped columns in nav_history — fixing automatically...")
    nav = nav.rename(columns={'nav': 'amfi_code', 'amfi_code': 'nav'})
    print("✅ Columns corrected: amfi_code and nav are now in correct order")
nav['amfi_code'] = nav['amfi_code'].astype(int)
nav['nav']       = pd.to_numeric(nav['nav'], errors='coerce')
print(f"   amfi_code sample: {nav['amfi_code'].unique()[:3].tolist()}  (should be ~100000–200000)")
print(f"   nav sample      : {nav['nav'].head(3).tolist()}  (should be small floats)")

print("\n✅ Datasets loaded!")
print(f"   nav         : {nav.shape}")
print(f"   fund_master : {fund_master.shape}")
print(f"   perf        : {perf.shape}")
print(f"   benchmark   : {benchmark.shape}")
print(f"\nBenchmark indices available: {benchmark['index_name'].unique()}")


---
## 📊 Step 1 — Compute Daily Returns
`daily_return = NAV_t / NAV_t-1 - 1` for all 40 schemes

In [ ]:
# Pivot NAV to wide format (date × amfi_code)
nav_pivot = nav.pivot_table(index='date', columns='amfi_code', values='nav')
nav_pivot = nav_pivot.sort_index()

# ✅ Fix: ensure amfi_code columns are int (pivot can make them float)
nav_pivot.columns = nav_pivot.columns.astype(int)
fund_master['amfi_code'] = fund_master['amfi_code'].astype(int)

# Daily returns
returns = nav_pivot.pct_change()
returns = returns.dropna(how='all')

# Validate distribution looks reasonable
mean_ret = returns.mean().mean() * 252 * 100  # Annualized %
daily_min = returns.min().min() * 100
daily_max = returns.max().max() * 100
print(f"✅ Returns computed: {returns.shape[0]} days × {returns.shape[1]} funds")
print(f"   Average annualised return across all funds: {mean_ret:.2f}%")
print(f"   Return range (daily): {daily_min:.3f}% to {daily_max:.3f}%")

# Plot return distribution for one fund
sample_code = int(returns.columns[0])
matches = fund_master[fund_master['amfi_code'] == sample_code]['scheme_name'].values
sample_name = matches[0] if len(matches) > 0 else str(sample_code)

plt.figure(figsize=(10, 4))
returns[sample_code].dropna().hist(bins=80, color='#1976D2', edgecolor='white', alpha=0.8)
plt.axvline(returns[sample_code].mean(), color='red', linestyle='--', label=f"Mean: {returns[sample_code].mean()*100:.3f}%")
plt.title(f'Daily Return Distribution — {sample_name[:40]}', fontsize=13, fontweight='bold')
plt.xlabel('Daily Return'); plt.ylabel('Frequency')
plt.legend(); plt.tight_layout()
plt.savefig('reports/chart_11_return_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved → reports/chart_11_return_distribution.png")


---
## 📈 Step 2 — Compute CAGR (1yr, 3yr, 5yr)
`CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

In [ ]:
def compute_cagr(nav_series, years):
    """Compute CAGR for given number of years."""
    nav_clean = nav_series.dropna()
    if len(nav_clean) == 0:
        return np.nan
    end_date   = nav_clean.index.max()
    start_date = end_date - pd.DateOffset(years=years)
    nav_end    = nav_clean.iloc[-1]
    before_start = nav_clean[nav_clean.index <= start_date]
    if len(before_start) == 0:
        return np.nan
    nav_start  = before_start.iloc[-1]
    if nav_start <= 0:
        return np.nan
    return (nav_end / nav_start) ** (1 / years) - 1

cagr_records = []
for code in nav_pivot.columns:
    name = fund_master[fund_master['amfi_code'] == code]['scheme_name'].values
    name = name[0] if len(name) > 0 else str(code)
    house = fund_master[fund_master['amfi_code'] == code]['fund_house'].values
    house = house[0] if len(house) > 0 else ''

    cagr_records.append({
        'amfi_code'  : code,
        'scheme_name': name,
        'fund_house' : house,
        'cagr_1yr'   : compute_cagr(nav_pivot[code], 1) * 100,
        'cagr_3yr'   : compute_cagr(nav_pivot[code], 3) * 100,
        'cagr_5yr'   : compute_cagr(nav_pivot[code], 5) * 100,
    })

cagr_df = pd.DataFrame(cagr_records)
print("✅ CAGR computed for all funds")
print("\nTop 5 funds by 3-Year CAGR:")
print(cagr_df.nlargest(5, 'cagr_3yr')[['scheme_name','cagr_1yr','cagr_3yr','cagr_5yr']].to_string(index=False))


---
## ⚡ Step 3 — Sharpe Ratio
`Sharpe = (Rp - Rf) / Std(Rp) × √252`  |  Rf = 6.5% (RBI repo rate)

In [ ]:
RF_ANNUAL = 0.065           # 6.5% RBI repo rate
RF_DAILY  = RF_ANNUAL / 252 # Daily risk-free rate

sharpe_records = []
for code in returns.columns:
    r = returns[code].dropna()
    if len(r) < 30:
        continue
    excess_return  = r - RF_DAILY
    sharpe         = (excess_return.mean() / r.std()) * np.sqrt(252)
    sharpe_records.append({'amfi_code': code, 'sharpe_ratio': sharpe})

sharpe_df = pd.DataFrame(sharpe_records)
print("✅ Sharpe Ratio computed for all funds")
print("\nTop 5 Funds by Sharpe Ratio:")
top_sharpe = sharpe_df.merge(fund_master[['amfi_code','scheme_name']], on='amfi_code')
print(top_sharpe.nlargest(5, 'sharpe_ratio')[['scheme_name','sharpe_ratio']].to_string(index=False))


---
## 📉 Step 4 — Sortino Ratio
Same as Sharpe but denominator uses only **downside** standard deviation (negative return days)

In [ ]:
sortino_records = []
for code in returns.columns:
    r = returns[code].dropna()
    if len(r) < 30:
        continue
    excess_return   = r - RF_DAILY
    downside_returns = r[r < 0]
    if len(downside_returns) == 0:
        continue
    downside_std    = downside_returns.std()
    sortino         = (excess_return.mean() / downside_std) * np.sqrt(252)
    sortino_records.append({'amfi_code': code, 'sortino_ratio': sortino})

sortino_df = pd.DataFrame(sortino_records)
print("✅ Sortino Ratio computed for all funds")
print("\nTop 5 Funds by Sortino Ratio:")
top_sortino = sortino_df.merge(fund_master[['amfi_code','scheme_name']], on='amfi_code')
print(top_sortino.nlargest(5, 'sortino_ratio')[['scheme_name','sortino_ratio']].to_string(index=False))


---
## 🎯 Step 5 — Alpha & Beta (OLS Regression)
OLS regression of fund returns on Nifty 100 returns using `scipy.stats.linregress`. Alpha = intercept × 252

In [ ]:
# Prepare benchmark returns (Nifty 100)
bench_names = benchmark['index_name'].unique()
print(f"Available benchmarks: {bench_names}")

# Use Nifty 100 if available, else first available
nifty100_name = None
for name in bench_names:
    if '100' in name.upper():
        nifty100_name = name
        break
if nifty100_name is None:
    nifty100_name = bench_names[0]
print(f"Using benchmark: {nifty100_name}")

bench_pivot = benchmark[benchmark['index_name'] == nifty100_name].set_index('date')['close_value']
bench_returns = bench_pivot.pct_change().dropna()

ab_records = []
for code in returns.columns:
    fund_ret  = returns[code].dropna()
    aligned   = pd.DataFrame({'fund': fund_ret, 'bench': bench_returns}).dropna()
    if len(aligned) < 60:
        continue
    slope, intercept, r_val, p_val, std_err = stats.linregress(aligned['bench'], aligned['fund'])
    beta  = slope
    alpha = intercept * 252  # Annualized alpha
    name  = fund_master[fund_master['amfi_code'] == code]['scheme_name'].values
    house = fund_master[fund_master['amfi_code'] == code]['fund_house'].values
    ab_records.append({
        'amfi_code'  : code,
        'scheme_name': name[0] if len(name) > 0 else str(code),
        'fund_house' : house[0] if len(house) > 0 else '',
        'alpha'      : round(alpha * 100, 4),
        'beta'       : round(beta, 4),
        'r_squared'  : round(r_val**2, 4),
        'benchmark'  : nifty100_name
    })

alpha_beta_df = pd.DataFrame(ab_records)

# Save alpha_beta.csv
alpha_beta_df.to_csv('alpha_beta.csv', index=False)
print(f"\n✅ alpha_beta.csv saved — {len(alpha_beta_df)} funds")
print("\nTop 5 Funds by Alpha (outperformance):")
print(alpha_beta_df.nlargest(5, 'alpha')[['scheme_name','alpha','beta','r_squared']].to_string(index=False))


---
## 📉 Step 6 — Maximum Drawdown
`max_drawdown = min(NAV / running_max - 1)` — Worst peak-to-trough loss

In [ ]:
dd_records = []
for code in nav_pivot.columns:
    nav_series = nav_pivot[code].dropna()
    if len(nav_series) < 30:
        continue
    running_max   = nav_series.cummax()
    drawdown      = (nav_series / running_max) - 1
    max_dd        = drawdown.min()
    max_dd_date   = drawdown.idxmin()
    # Find peak before max drawdown
    peak_date     = nav_series[:max_dd_date].idxmax()
    name          = fund_master[fund_master['amfi_code'] == code]['scheme_name'].values
    dd_records.append({
        'amfi_code'   : code,
        'scheme_name' : name[0] if len(name) > 0 else str(code),
        'max_drawdown': round(max_dd * 100, 2),
        'peak_date'   : str(peak_date.date()),
        'trough_date' : str(max_dd_date.date()),
    })

drawdown_df = pd.DataFrame(dd_records)
print("✅ Maximum Drawdown computed for all funds")
print("\nWorst 5 Drawdowns:")
print(drawdown_df.nsmallest(5, 'max_drawdown')[['scheme_name','max_drawdown','peak_date','trough_date']].to_string(index=False))


---
## 🏆 Step 7 — Fund Scorecard (0–100)
Composite: **30%** × 3yr CAGR rank + **25%** × Sharpe rank + **20%** × Alpha rank + **15%** × Expense ratio rank (inverse) + **10%** × Max DD rank (inverse)

In [ ]:
# Merge all metrics
scorecard = (cagr_df[['amfi_code','scheme_name','fund_house','cagr_3yr']]
             .merge(sharpe_df, on='amfi_code', how='left')
             .merge(sortino_df, on='amfi_code', how='left')
             .merge(alpha_beta_df[['amfi_code','alpha','beta','r_squared']], on='amfi_code', how='left')
             .merge(drawdown_df[['amfi_code','max_drawdown','peak_date','trough_date']], on='amfi_code', how='left')
             .merge(fund_master[['amfi_code','expense_ratio_pct']], on='amfi_code', how='left'))

scorecard = scorecard.dropna(subset=['cagr_3yr','sharpe_ratio','alpha','max_drawdown'])

n = len(scorecard)

# Rank each metric (higher rank = better)
scorecard['rank_3yr_cagr']     = scorecard['cagr_3yr'].rank(ascending=True)
scorecard['rank_sharpe']       = scorecard['sharpe_ratio'].rank(ascending=True)
scorecard['rank_alpha']        = scorecard['alpha'].rank(ascending=True)
scorecard['rank_expense']      = scorecard['expense_ratio_pct'].rank(ascending=False)  # Lower is better
scorecard['rank_max_dd']       = scorecard['max_drawdown'].rank(ascending=False)        # Less negative is better

# Composite Score 0-100
scorecard['score'] = (
    0.30 * (scorecard['rank_3yr_cagr'] / n * 100) +
    0.25 * (scorecard['rank_sharpe']   / n * 100) +
    0.20 * (scorecard['rank_alpha']    / n * 100) +
    0.15 * (scorecard['rank_expense']  / n * 100) +
    0.10 * (scorecard['rank_max_dd']   / n * 100)
).round(2)

scorecard = scorecard.sort_values('score', ascending=False).reset_index(drop=True)
scorecard.index = scorecard.index + 1  # Rank starts from 1

# Save fund_scorecard.csv
cols_to_save = ['amfi_code','scheme_name','fund_house','cagr_3yr','sharpe_ratio',
                'sortino_ratio','alpha','beta','max_drawdown','expense_ratio_pct','score']
scorecard[cols_to_save].to_csv('fund_scorecard.csv', index=True, index_label='rank')

print("✅ fund_scorecard.csv saved!")
print(f"\n🏆 TOP 10 FUNDS BY SCORECARD:")
print(scorecard[['scheme_name','cagr_3yr','sharpe_ratio','alpha','max_drawdown','score']].head(10).to_string())

# Bar chart of top 10 scores
plt.figure(figsize=(13, 6))
top10 = scorecard.head(10)
bars = plt.barh(top10['scheme_name'].str[:35][::-1], top10['score'][::-1],
                color=plt.cm.RdYlGn(np.linspace(0.4, 1.0, 10)), edgecolor='white')
plt.xlabel('Composite Score (0–100)', fontsize=12)
plt.title('🏆 Fund Scorecard — Top 10 Funds', fontsize=14, fontweight='bold')
for bar, score in zip(bars, top10['score'][::-1]):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{score:.1f}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/chart_12_fund_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved → reports/chart_12_fund_scorecard.png")


---
## 📊 Step 8 — Benchmark Comparison Chart
Top 5 funds vs Nifty 50 and Nifty 100 over 3 years. Compute tracking error.

In [ ]:
# Get top 5 fund codes from scorecard
top5_codes = scorecard.head(5)['amfi_code'].tolist()
top5_names = scorecard.head(5)['scheme_name'].str[:30].tolist()

# 3-year cutoff date
end_date   = nav['date'].max()
start_date = end_date - pd.DateOffset(years=3)

# Filter NAV for top 5 funds
nav_top5 = nav_pivot[top5_codes]
nav_top5 = nav_top5[nav_top5.index >= start_date]

# Normalise to 100 at start
nav_norm = nav_top5.div(nav_top5.iloc[0]) * 100

fig = go.Figure()

colors = ['#1976D2','#43A047','#E53935','#FB8C00','#8E24AA']

for i, (code, name) in enumerate(zip(top5_codes, top5_names)):
    fig.add_trace(go.Scatter(
        x=nav_norm.index.astype(str), y=nav_norm[code],
        mode='lines', name=name,
        line=dict(color=colors[i], width=2)
    ))

# Add Nifty 50 benchmark
for idx_name, color, dash in [
    ('NIFTY 50',  '#212121', 'dash'),
    (nifty100_name, '#757575', 'dot')
]:
    if idx_name in benchmark['index_name'].values:
        b = benchmark[benchmark['index_name'] == idx_name].set_index('date')['close_value']
        b = b[b.index >= start_date]
        if len(b) > 0:
            b_norm = b / b.iloc[0] * 100
            fig.add_trace(go.Scatter(
                x=b_norm.index.astype(str), y=b_norm,
                mode='lines', name=idx_name,
                line=dict(color=color, width=2.5, dash=dash)
            ))

fig.update_layout(
    title='📊 Top 5 Funds vs Benchmark (3-Year, Normalised to 100)',
    xaxis_title='Date', yaxis_title='Normalised Value (Base=100)',
    template='plotly_white', height=550, title_font_size=15,
    legend=dict(orientation='v', x=1.01)
)
fig.write_image('reports/chart_13_benchmark_comparison.png', width=1400, height=550, scale=1.5)
fig.show()
print("✅ Saved → reports/chart_13_benchmark_comparison.png")

# Tracking Error
print("\n📐 Tracking Error (Annualised):")
nifty_ret = bench_returns.copy()
for code, name in zip(top5_codes, top5_names):
    fund_ret_series = returns[code].dropna()
    aligned_te = pd.DataFrame({'fund': fund_ret_series, 'bench': nifty_ret}).dropna()
    if len(aligned_te) > 0:
        tracking_error = (aligned_te['fund'] - aligned_te['bench']).std() * np.sqrt(252) * 100
        print(f"   {name[:40]:40s}: {tracking_error:.2f}%")


---
## 📋 Step 9 — Full Comparison Table

In [ ]:
print("\n📋 COMPLETE FUND COMPARISON TABLE")
print("="*100)
display_cols = ['scheme_name','fund_house','cagr_3yr','sharpe_ratio','sortino_ratio',
                'alpha','beta','max_drawdown','expense_ratio_pct','score']
display_df = scorecard[display_cols].head(20).round(3)
display_df.columns = ['Fund','House','CAGR_3yr%','Sharpe','Sortino',
                      'Alpha%','Beta','MaxDD%','ExpRatio%','Score']
print(display_df.to_string())
print("\n✅ All performance metrics computed and saved!")
print("   → fund_scorecard.csv")
print("   → alpha_beta.csv")
print("   → reports/chart_12_fund_scorecard.png")
print("   → reports/chart_13_benchmark_comparison.png")


---
*Performance Analytics completed — Day 4 | Bluestock MF Analytics Internship*